- 1️⃣ Read Bronze tables
- 2️⃣ Apply cleaning (dedup + null validation)
- 3️⃣ Track DQ metrics
- 4️⃣ Write Silver tables
- 5️⃣ Save DQ audit logs into table

⭐ WHY DQ LOGGING?

- Real pipelines must answer:
- 
- How many rows came?
- 
- How many removed?
- 
- Why data changed?
- 
- This is essential for production monitoring.

In [0]:
from pyspark.sql.functions import col, current_timestamp


In [0]:
silver_config = {

   "orders": {
       "primary_key": ["order_id"],
       "not_null": ["order_id","customer_id"]
   },

   "order_items": {
       "primary_key": ["order_id","order_item_id"],
       "not_null": ["order_id","product_id"]
   },

   "order_payments": {
       "primary_key": ["order_id","payment_sequential"],
       "not_null": ["order_id"]
   },

   "order_reviews": {
       "primary_key": ["review_id"],
       "not_null": ["review_id"]
   }

}


In [0]:
def load_bronze(table_name):

    return spark.table(f"e_comm_databricks.pipeline_bronze.{table_name}")


In [0]:
def deduplicate(df, keys):

    before = df.count()

    df = df.dropDuplicates(keys)

    after = df.count()

    removed = before - after

    return df, removed


In [0]:
def remove_nulls(df, columns):

    before = df.count()

    for c in columns:
        df = df.filter(col(c).isNotNull())

    after = df.count()

    removed = before - after

    return df, removed


In [0]:
def write_silver(df, table_name):

    df.write \
      .format("delta") \
      .mode("overwrite") \
      .saveAsTable(f"e_comm_databricks.pipeline_silver.{table_name}")


In [0]:
dq_logs = []


In [0]:
def process_table(table_name, rules):

    df = load_bronze(table_name)

    initial_count = df.count()

    df, dup_removed = deduplicate(df, rules["primary_key"])

    df, null_removed = remove_nulls(df, rules["not_null"])

    final_count = df.count()

    write_silver(df, table_name)

    dq_logs.append(
        (table_name, initial_count, dup_removed, null_removed, final_count)
    )


In [0]:
for table, rules in silver_config.items():

    process_table(table, rules)


In [0]:
dq_df = spark.createDataFrame(

    dq_logs,

    ["table_name","rows_before","duplicates_removed","nulls_removed","rows_after"]

).withColumn("run_timestamp", current_timestamp())


In [0]:
dq_df.write \
 .format("delta") \
 .mode("append") \
 .saveAsTable("e_comm_databricks.pipeline_silver.dq_audit_log")
